# mse-reconstruction-loss — worked example 3: Compare MSE and BCE reconstruction loss on binary (0/1) image batches

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

For autoencoder reconstructions of binary (black-and-white) images, both MSE and binary cross-entropy (BCE) are common choices. MSE treats each pixel error as a squared distance, while BCE treats the reconstruction as a Bernoulli probability and penalizes confident wrong predictions more harshly. This worked example computes both losses and shows that BCE is more sensitive to errors on near-0 or near-1 pixels.

## Worked solution

**Step 1 — create a batch of binary targets.** We use `torch.randint(0, 2, ...)` to generate 0/1 images and `.float()` to convert to float32, as loss functions expect float.

**Step 2 — simulate a "sigmoid output" decoder.** In a VAE with binary data, the decoder's last activation is Sigmoid, so outputs are in `(0, 1)`. We add small noise to the targets and pass through `torch.sigmoid` to get predictions in the valid range.

**Step 3 — compute MSE.** `F.mse_loss(pred, target)` — straightforward.

**Step 4 — compute BCE.** `F.binary_cross_entropy(pred, target)` — expects pred in `(0, 1)` and target in `{0, 1}`. Do NOT use `binary_cross_entropy_with_logits` here since we already applied sigmoid.

**Step 5 — compare.** Print both losses. For a perfect reconstruction, BCE would be near 0 (high confidence, correct). For a random reconstruction, BCE ≈ 0.693 (log 2) while MSE ≈ 0.25 (mean of `(0.5 - target)²`).

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def compare_mse_bce(target: torch.Tensor, pred: torch.Tensor) -> dict:
    """
    Compute both MSE and BCE reconstruction losses.
    target: binary float tensor (values 0.0 or 1.0)
    pred:   float tensor in (0, 1) -- sigmoid output
    Returns dict with 'mse' and 'bce' scalar tensors.
    """
    mse = F.mse_loss(pred, target)
    bce = F.binary_cross_entropy(pred, target)
    return {'mse': mse, 'bce': bce}

# Case 1: near-perfect reconstruction
torch.manual_seed(42)
B, H, W = 6, 8, 8
target = torch.randint(0, 2, (B, 1, H, W)).float()
# Pred very close to target: sigmoid(10*(2*t-1)) is ~0.9999 where t=1, ~0.0001 where t=0
pred_good = torch.sigmoid(10.0 * (2.0 * target - 1.0))
losses_good = compare_mse_bce(target, pred_good)
print(f"Near-perfect -> MSE: {losses_good['mse'].item():.5f}, BCE: {losses_good['bce'].item():.5f}")

# Case 2: random (chance) reconstruction
pred_rand = torch.full_like(target, 0.5)  # always predict 0.5
losses_rand = compare_mse_bce(target, pred_rand)
print(f"Random (0.5) -> MSE: {losses_rand['mse'].item():.5f}, BCE: {losses_rand['bce'].item():.5f}")

assert losses_good['bce'].item() < losses_rand['bce'].item(), "near-perfect should have lower BCE"
assert losses_good['mse'].item() < losses_rand['mse'].item(), "near-perfect should have lower MSE"
print("Loss comparison assertions passed!")